In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv("/content/birth_data.csv")
cols = ["male","female","total"]

for c in cols:
    df[c] = df[c].astype(str).str.replace(",","").astype(float)

df["month"] = pd.to_datetime(df["month"])
df = df.sort_values("month")

In [ ]:
lags = [1,2,3,12]

for lag in lags:
    df[f"male_lag{lag}"] = df["male"].shift(lag)
    df[f"female_lag{lag}"] = df["female"].shift(lag)

df["month_num"] = df["month"].dt.month
df["year"] = df["month"].dt.year

df = df.dropna().reset_index(drop=True)

In [ ]:
features = [c for c in df.columns if "lag" in c] + ["month_num","year"]

X = df[features]

y_male = df["male"]
y_female = df["female"]

In [ ]:
rf_m = RandomForestRegressor(n_estimators=200, random_state=42)
xgb_m = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5)
lr_m = LinearRegression()

rf_f = RandomForestRegressor(n_estimators=200, random_state=42)
xgb_f = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5)
lr_f = LinearRegression()

In [ ]:
rf_m.fit(X,y_male)
xgb_m.fit(X,y_male)
lr_m.fit(X,y_male)

rf_f.fit(X,y_female)
xgb_f.fit(X,y_female)
lr_f.fit(X,y_female)

LinearRegression()

In [ ]:
def ensemble_predict(X_input):

    male_pred = (
        0.4 * xgb_m.predict(X_input) +
        0.3 * rf_m.predict(X_input) +
        0.3 * lr_m.predict(X_input)
    )

    female_pred = (
        0.4 * xgb_f.predict(X_input) +
        0.3 * rf_f.predict(X_input) +
        0.3 * lr_f.predict(X_input)
    )

    total_pred = male_pred + female_pred

    return male_pred[0], female_pred[0], total_pred[0]

In [ ]:
def predict_until(target_month):

    data = df.copy()
    target_month = pd.to_datetime(target_month)

    while data["month"].max() < target_month:

        last = data.iloc[-1:]

        next_month = last["month"].iloc[0] + pd.DateOffset(months=1)

        row_features = last[features].fillna(0)

        male_pred, female_pred, total_pred = ensemble_predict(row_features)

        new_row = {
            "month": next_month,
            "male": male_pred,
            "female": female_pred,
            "total": total_pred
        }

        data = pd.concat([data, pd.DataFrame([new_row])], ignore_index=True)

    return data[data["month"] == target_month]

In [ ]:
target = input("Enter month to predict (YYYY-MM): ")

result = predict_until(target)

print(result[["month","male","female","total"]])

Enter month to predict (YYYY-MM): 2026-04
         month          male        female          total
279 2026-04-01  95330.032611  85593.958712  180923.991323


In [ ]:
pip install joblib

In [ ]:
import joblib

joblib.dump(rf_m,"rf_m.pkl")
joblib.dump(xgb_m,"xgb_m.pkl")
joblib.dump(lr_m,"lr_m.pkl")

joblib.dump(rf_f,"rf_f.pkl")
joblib.dump(xgb_f,"xgb_f.pkl")
joblib.dump(lr_f,"lr_f.pkl")

['lr_f.pkl']

In [ ]:
from google.colab import drive
drive.mount('/content/drive')